# CLASS 2025 Daisyworld Experiment
## Investigating the importance of arable land proportion $p$



In [ ]:
from copy import deepcopy
import os

import matplotlib
import matplotlib.pyplot as plt

import numpy as np
import sympy as sp
import pandas as pd
from dw.simple_dw import SimpleDaisyWorld

# saving data and figures
save_images = True
save_data = True
experiment_folder = "class_2025_p"
frames_folder = "frames"
p_data_filename = "class_2025_data"
ap_data_filename = "class_2025_ap_map"

In [ ]:
plt.rcParams["font.size"] = 16

In [ ]:
world = SimpleDaisyWorld()
# Modify variables

## albedo of light and dark daisies
albedo_light_daisies = 0.75
albedo_dark_daisies = 0.25
albedo_ground = 0.5

## starting populations of daisies, in proportion of ground covered
cover_light_daisies = 0.2
cover_dark_daisies = 0.2
cover_arable_ground = 1.0

## set the albedo for light daisise
world.set_Al(albedo_light_daisies)
world.set_Ad(albedo_dark_daisies)
world.set_Ag(albedo_ground)

## set the values for initial ground cover
world.set_p(cover_arable_ground)
world.set_initial_ad(cover_dark_daisies)
world.set_initial_al(cover_light_daisies)

world.steps_per_period = 6000

world.run_sim()
fig, ax = world.plot_curve(show_habitable=True)
fig.suptitle("Daisyworld, typical run")
if save_images:
    plt.savefig(os.path.join(experiment_folder, "daisyworld_defaults.png"))
plt.show()

In [ ]:
# experiment: varying arable land proportion _p_ 
# questions: 
# * does the ability of daisyworld to regulate temperature within a habitable range decrease monotonically with p?
# * is the relationship between habitable temperature duration and _p_ linear?
# * is there a threshold for p (above 0) where daisyworl

# resolution of arable land variable p
d_arable = 0.01


# arable land values
exp_arable_land = []
# live daisy duration
exp_persistence_ad = []
exp_persistence_al = []
exp_habitable = []

for p_index, arable in enumerate(np.arange(0.0, 1.0 +d_arable, d_arable)):
    world = SimpleDaisyWorld()
    world.set_p(arable)
    
    world.set_initial_ad(0.2* world.p)
    world.set_initial_al(0.2* world.p)
    #world.initial_ag = world.p - world.initial_ab - world.initial_aw
    
    world.reset()
    world.run_sim()
    fig, ax = world.plot_curve(show_habitable=True)
    fig.suptitle(f"Daisyworld with {world.p} arable land")
    
    if save_images:
        if os.path.exists(os.path.join(experiment_folder, frames_folder)):
            pass
        else:
            os.mkdir(os.path.join(experiment_folder, frames_folder))

        fig.savefig(os.path.join(experiment_folder, frames_folder, f"frame_{p_index}_experiment.png"))
        
    plt.show()
    plt.close()
    
    exp_persistence_ad.append((np.array(world.list_ab) > 0.001).sum())
    exp_persistence_al.append((np.array(world.list_aw) > 0.001).sum())
    cool_enough = (1.0 * (np.array(world.list_Te) <= world.Toptim + world.g**(-1/2)))
    warm_enough = (1.0 * (np.array(world.list_Te) >= world.Toptim - world.g**(-1/2)))
    exp_habitable.append((cool_enough * warm_enough).sum())

    exp_arable_land.append(world.p)

# control condition
# arable land values
cont_arable_land = []
# live daisy duration
cont_persistence_ad = []
cont_persistence_al = []
cont_habitable = []

for p_index, arable in enumerate(np.arange(0.0, 1.0 +d_arable, d_arable)):
    print(p_index)
    world = SimpleDaisyWorld()
    world.set_p(arable)
    
    world.set_initial_ad(0.2* world.p)
    world.set_initial_al(0.2* world.p)
    #world.initial_ag = world.p - world.initial_ab - world.initial_aw
    
    world.set_Al(world.Ag)
    world.set_Ad(world.Ag)
    
    world.reset()
    world.run_sim()
    fig, ax = world.plot_curve(show_habitable=True)
    fig.suptitle(f"(neutral albedo) Daisyworld with {world.p} arable land")

    if save_images:
        if os.path.exists(os.path.join(experiment_folder, frames_folder)):
            pass
        else:
            os.mkdir(os.path.join(experiment_folder, frames_folder))

        fig.savefig(os.path.join(experiment_folder, frames_folder, f"frame_{p_index}_control.png"))
        
    plt.show()
    plt.close()
    cont_persistence_ad.append((np.array(world.list_ab) > 0.001).sum())
    cont_persistence_al.append((np.array(world.list_aw) > 0.001).sum())
    cool_enough = (1.0 * (np.array(world.list_Te) <= world.Toptim + world.g**(-1/2)))
    warm_enough = (1.0 * (np.array(world.list_Te) >= world.Toptim - world.g**(-1/2)))
    cont_habitable.append((cool_enough * warm_enough).sum())

    cont_arable_land.append(world.p)


In [ ]:
# put the results in a dataframe

df_exp = pd.DataFrame({\
    "control_arable_land_p": cont_arable_land, "experiment_arable_land_p": exp_arable_land,\
    "control_habitable": cont_habitable, "experiment_habitable": exp_habitable,\
    "control_persistence_ad": cont_persistence_ad, "control_persistence_al": cont_persistence_al,\
    "experiment_persistence_ad": exp_persistence_ad, "experiment_persistence_al": exp_persistence_al\
    })

#save data
if save_data:
    if os.path.exists(os.path.join(experiment_folder)):
        pass    
    else:
        os.mkdir(experiment_folder)
    
    df_exp.to_csv(os.path.join(experiment_folder, f"{p_data_filename}.csv"))

In [ ]:
load_data = True
if load_data:
    df_exp = pd.read_csv(os.path.join(experiment_folder, p_data_filename))
    

In [ ]:
control_cmap = plt.get_cmap("plasma")
ccolor_a = control_cmap(0.33)
ccolor_b = control_cmap(0.67)
exp_cmap = plt.get_cmap("viridis")
xcolor_a = exp_cmap(0.33)
xcolor_b = exp_cmap(0.67)

plt.figure(figsize=(8,5))
plt.plot(df_exp["control_arable_land_p"], df_exp["control_habitable"], "--", color=ccolor_b, lw=3, label="habitable duration (neutral albedo)")
plt.plot(df_exp["experiment_arable_land_p"], df_exp["experiment_habitable"], color=xcolor_b, lw=3, label="habitable duration (experiment)")
plt.axis((0,1., 0,4500))
plt.title("Habitable Temperature Duration")
plt.xlabel(r"Arable land proportion $p$")
plt.ylabel("Habitable longevity (time units "+r"$t$)")
plt.legend()


plt.figure(figsize=(8,5))
plt.plot(df_exp["control_arable_land_p"], df_exp["control_persistence_ad"], ".-", color=ccolor_a, lw=3, alpha=0.50, label="dark daisy persistence (neutral albedo)")
plt.plot(df_exp["control_arable_land_p"], df_exp["control_persistence_al"], "--", color=ccolor_b,  lw=2, alpha=0.50, label="daisy persistence (neutral albedo)")
plt.plot(df_exp["experiment_arable_land_p"], df_exp["experiment_persistence_ad"], lw=4,  color=xcolor_a, alpha=0.50, label="dark daisy persistence (experiment)")
plt.plot(df_exp["experiment_arable_land_p"], df_exp["experiment_persistence_al"], lw=3,  color=xcolor_b, alpha=0.50, label=f"light daisy persistence (experiment)")
plt.title("Daisy Persistence")
plt.axis((0,1.0, 0,4800))
plt.xlabel(r"Arable land proportion $p$")
plt.ylabel("Daisy persistence (time units "+r"$t$)")
plt.legend()
plt.show()

In [ ]:

plt.figure(figsize=(8,11))

plt.subplot(211)
plt.plot(df_exp["control_arable_land_p"], df_exp["control_habitable"], "--", color=ccolor_b, lw=3, label="habitable duration (neutral albedo)")
plt.plot(df_exp["experiment_arable_land_p"], df_exp["experiment_habitable"], color=xcolor_b, lw=3, label="habitable duration (experiment)")
plt.axis((0,1., 0,4500))
plt.title("Habitable Temperature Duration")
#plt.xlabel(r"Arable land proportion $p$")
plt.ylabel("Habitable longevity (time units "+r"$t$)")
plt.legend()


plt.subplot(212)
plt.plot(df_exp["control_arable_land_p"], df_exp["control_persistence_ad"], ".-", color=ccolor_a, lw=3, alpha=0.50, label="dark daisy persistence (neutral albedo)")
plt.plot(df_exp["control_arable_land_p"], df_exp["control_persistence_al"], "--", color=ccolor_b,  lw=2, alpha=0.50, label="daisy persistence (neutral albedo)")
plt.plot(df_exp["experiment_arable_land_p"], df_exp["experiment_persistence_ad"], lw=4,  color=xcolor_a, alpha=0.50, label="dark daisy persistence (experiment)")
plt.plot(df_exp["experiment_arable_land_p"], df_exp["experiment_persistence_al"], lw=3,  color=xcolor_b, alpha=0.50, label=f"light daisy persistence (experiment)")
plt.title("Daisy Persistence")
plt.axis((0,1.0, 0,4800))
plt.xlabel(r"Arable land proportion $p$")
plt.ylabel("Daisy persistence (time units "+r"$t$)")
plt.legend()

if save_images:
    plot_filename = "temperature_daisies_duration.png"
    plt.savefig(os.path.join(experiment_folder, plot_filename))
    
plt.show()

In [ ]:
# mapping out arable land p and albedo difference

# albedo can be at most a perfect reflector (1.0) or perfect black body (0.0)
da = 0.005
dp = 0.01
diff_albedo = np.arange(0.0, 0.5+da, da)
arable_p = np.arange(0.0, 1.0+dp, dp)

ap_map = np.zeros((arable_p.shape[0], diff_albedo.shape[0]))

for hp, arable in enumerate(arable_p):
    for wa, diff_a in enumerate(diff_albedo):
        
        world = SimpleDaisyWorld()
        world.set_p(arable)

        world.set_Ad(world.Ag - diff_a)
        world.set_Al(world.Ag + diff_a)
        world.set_initial_ad(0.2* world.p)
        world.set_initial_al(0.2* world.p)
        #world.initial_ag = world.p - world.initial_ab - world.initial_aw
        
        world.reset()
        world.run_sim()
       
        
        exp_persistence_ad.append((np.array(world.list_ab) > 0.001).sum())
        exp_persistence_al.append((np.array(world.list_aw) > 0.001).sum())
        cool_enough = (1.0 * (np.array(world.list_Te) <= world.Toptim + world.g**(-1/2)))
        warm_enough = (1.0 * (np.array(world.list_Te) >= world.Toptim - world.g**(-1/2)))
        exp_habitable.append((cool_enough * warm_enough).sum())
    
        exp_arable_land.append(world.p)

        ap_map[hp, wa] = (cool_enough * warm_enough).sum()


In [ ]:
if save_data:
    np.save(os.path.join(experiment_folder, f"{ap_data_filename}.npy"), ap_map)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(8,8))

ax.imshow(ap_map, cmap="magma")
ax.set_yticks(np.arange(0, arable_p.shape[0],16))
ax.set_xticks(np.arange(0, diff_albedo.shape[0],16))
ax.set_yticklabels(arable_p[::16]) 
ax.set_xticklabels(diff_albedo[::16]) 
ax.set_ylabel("Arable land")
ax.set_xlabel(r"+/-$\Delta$albedo")

ax.set_title("Daisyworld\nArable land/albedo margin map")

plt.show()